In [54]:
import numpy as np

from superfv.stencils import conservative_interpolation as ci
from superfv.sweep import stencil_sweep
from superfv.tools.norms import linf_norm

p = 3

domain = (0, 1)
N = 1024

h = (domain[1] - domain[0]) / N
cell_interfaces = np.linspace(domain[0], domain[1], N + 1)
cell_centers = 0.5 * (cell_interfaces[:-1] + cell_interfaces[1:])

In [55]:
# Method 1
x, _ = np.polynomial.legendre.leggauss(ci.n_gauss_legendre_nodes(p))
mesh1 = cell_centers[:, np.newaxis] + 0.5 * h * x

In [56]:
# Method 2
stencil_weights = ci.gauss_legendre_nodes(p)
ninterps, stencil_size = stencil_weights.shape
stencil_reach = (stencil_size - 1) // 2

_N_ = N + 2 * stencil_reach
_cell_interfaces_ = np.linspace(
    domain[0] - stencil_reach * h, domain[1] + stencil_reach * h, _N_ + 1
)
_cell_centers_flat_ = 0.5 * (_cell_interfaces_[:-1] + _cell_interfaces_[1:])
_cell_centers_ = _cell_centers_flat_.reshape(1, _N_, 1, 1, 1)

_mesh2_ = np.empty((1, _N_, 1, 1, ninterps))
stencil_sweep(_cell_centers_, stencil_weights, _mesh2_, "x")
mesh2 = _mesh2_[0, slice(stencil_reach, -stencil_reach), 0, 0, :]

In [57]:
linf_norm(mesh1 - mesh2)

2.220446049250313e-16